In [17]:
DATA = "bookings.jsonl"

# Task 2: Contracts, nested data and a star schema

In [18]:
import json
import re
from datetime import date, datetime
from typing import Literal, Optional

import duckdb
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

## Part A: The contract

In [19]:
class Ticket(BaseModel):
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    id: str
    name: str
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]

    @field_validator("id")
    @classmethod
    def check_id(cls, v):
        # two digits, a dash, four digits
        if not re.fullmatch(r"\d{2}-\d{4}", v):
            raise ValueError("student id must look like 58-1234")
        return v


class Event(BaseModel):
    id: str
    title: str
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: date
    venue: str


class Booking(BaseModel):
    booking_id: str
    booked_at: datetime
    student: Student
    event: Event                      # required, so null / missing fails
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]
    promo_code: Optional[str] = None  # the drifted field, not in older documents

    @model_validator(mode="after")
    def total_matches_tickets(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if abs(expected - self.total) > 0.001:
            raise ValueError("total does not match sum of qty x price")
        return self

    @model_validator(mode="after")
    def booked_before_event(self):
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked_at is not before the event date")
        return self

### Validate every line, then drop duplicates

In [20]:
def short_reason(err: ValidationError) -> str:
    """Turn the first pydantic error into one readable reason."""
    e = err.errors()[0]
    where = ".".join(str(p) for p in e["loc"] if not isinstance(p, int))
    msg = e["msg"].replace("Value error, ", "")
    return f"{where}: {msg}" if where else msg


lines_read = 0
valid = []        # Booking objects that passed the contract
quarantine = []   # {"booking_id", "reason"}

with open(DATA) as f:
    for line in f:
        if not line.strip():
            continue
        lines_read += 1
        try:
            doc = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"booking_id": None, "reason": "invalid json"})
            continue
        try:
            valid.append(Booking.model_validate(doc))
        except ValidationError as err:
            quarantine.append({"booking_id": doc.get("booking_id"), "reason": short_reason(err)})

# duplicates are checked only among bookings that already passed
seen = set()
bookings_ok = []
for b in valid:
    if b.booking_id in seen:
        quarantine.append({"booking_id": b.booking_id, "reason": "duplicate booking_id"})
    else:
        seen.add(b.booking_id)
        bookings_ok.append(b)

with open("quarantine.jsonl", "w") as f:
    for q in quarantine:
        f.write(json.dumps(q) + "\n")

In [21]:
print("lines read:  ", lines_read)
print("passed:      ", len(bookings_ok))
print("quarantined: ", len(quarantine))
print()
pd.DataFrame(quarantine)["reason"].value_counts().rename("count").to_frame()

lines read:   283
passed:       259
quarantined:  24



,count
reason,
student.id: student id must look like 58-1234,4
total does not match sum of qty x price,4
tickets.qty: Input should be greater than or equal to 1,3
"tickets.type: Input should be 'standard', 'vip' or 'student'",3
duplicate booking_id,3
booked_at is not before the event date,3
event: Input should be a valid dictionary or instance of Event,2
"tickets: List should have at least 1 item after validation, not 0",2


In [22]:
pd.DataFrame(quarantine)

,booking_id,reason
0,B0011,"tickets.type: Input should be 'standard', 'vip..."
1,B0016,tickets.qty: Input should be greater than or e...
2,B0024,student.id: student id must look like 58-1234
3,B0049,event: Input should be a valid dictionary or i...
4,B0057,student.id: student id must look like 58-1234
5,B0060,student.id: student id must look like 58-1234
6,B0090,"tickets.type: Input should be 'standard', 'vip..."
7,B0101,total does not match sum of qty x price
8,B0106,student.id: student id must look like 58-1234
9,B0116,booked_at is not before the event date


## Part B: Flatten

In [23]:
booking_rows = []
ticket_rows = []

for b in bookings_ok:
    booking_rows.append({
        "booking_id": b.booking_id,
        "booked_at": b.booked_at,
        "student_id": b.student.id,
        "student_name": b.student.name,
        "student_faculty": b.student.faculty,
        "event_id": b.event.id,
        "event_title": b.event.title,
        "event_club": b.event.club,
        "event_date": b.event.date,
        "event_venue": b.event.venue,
        "total": b.total,
        "payment": b.payment,
        "status": b.status,
        "promo_code": b.promo_code,
    })
    for t in b.tickets:
        ticket_rows.append({
            "booking_id": b.booking_id,
            "type": t.type,
            "qty": t.qty,
            "price": t.price,
            "amount": t.qty * t.price,
        })

bookings = pd.DataFrame(booking_rows)
booking_tickets = pd.DataFrame(ticket_rows)

# store booked_at as plain UTC timestamps (no tz) so DuckDB doesn't shift them
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True).dt.tz_convert(None)
bookings["event_date"] = pd.to_datetime(bookings["event_date"])

print(bookings.shape, booking_tickets.shape)
bookings.head()

(259, 14) (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,None
1,B0002,2026-09-27 12:29:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,None
2,B0003,2026-10-13 20:19:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,None
3,B0004,2026-11-01 06:42:00,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,105.0,card,confirmed,CLUB10
4,B0005,2026-10-28 19:06:00,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,20.0,cash,confirmed,None


In [24]:
booking_tickets.head()

,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


In [25]:
# checks
assert bookings["booking_id"].is_unique

per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"].sub(per_booking).abs()
assert (check < 0.001).all(), "ticket amounts don't add up to total"
print("both checks passed")

both checks passed


### Schema drift

In [26]:
# compare every raw document's keys against the fields in the README example
expected_keys = {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"}

raw = []
with open(DATA) as f:
    for line in f:
        if line.strip():
            raw.append(json.loads(line))

new_fields = set()
for doc in raw:
    new_fields |= set(doc) - expected_keys
print("fields not in the example:", new_fields)

# look at it only on the valid bookings (bad rows, e.g. booked after the event, would blur the picture)
valid_ids = set(bookings["booking_id"])
for field in new_fields:
    first_copy = {}
    for d in raw:
        if d.get("booking_id") in valid_ids and d["booking_id"] not in first_copy:
            first_copy[d["booking_id"]] = d
    rows = pd.DataFrame([
        {"booked_at": pd.to_datetime(d["booked_at"]), "has_field": field in d}
        for d in first_copy.values()
    ]).sort_values("booked_at")
    first_with = rows.loc[rows["has_field"], "booked_at"].min()
    last_without = rows.loc[~rows["has_field"], "booked_at"].max()
    print(f"\n{field}: {rows['has_field'].sum()} valid bookings have it, {(~rows['has_field']).sum()} don't")
    print("  first booking that has it:     ", first_with)
    print("  last booking that doesn't:     ", last_without)
    print("  clean cut-over:", last_without < first_with)


fields not in the example: {'promo_code'}

promo_code: 173 valid bookings have it, 86 don't
  first booking that has it:      2026-10-20 01:05:00+00:00
  last booking that doesn't:      2026-10-19 18:43:00+00:00
  clean cut-over: True


## Part C: Star schema in DuckDB

In [27]:
con = duckdb.connect()
con.register("bookings", bookings)
con.register("booking_tickets", booking_tickets)

con.execute("""
    CREATE OR REPLACE TABLE dim_event AS
    SELECT ROW_NUMBER() OVER (ORDER BY event_id) AS event_key, *
    FROM (
        SELECT DISTINCT event_id, event_title, event_club, CAST(event_date AS DATE) AS event_date, event_venue
        FROM bookings
    )
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_student AS
    SELECT ROW_NUMBER() OVER (ORDER BY student_id) AS student_key, *
    FROM (
        SELECT DISTINCT student_id, student_name, student_faculty
        FROM bookings
    )
""")

con.execute("""
    CREATE OR REPLACE TABLE fact_ticket_sales AS
    SELECT
        t.booking_id,
        e.event_key,
        s.student_key,
        t.type   AS ticket_type,
        t.qty,
        t.price,
        t.amount,
        b.status,
        CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS date_key
    FROM booking_tickets t
    JOIN bookings b    ON b.booking_id = t.booking_id
    JOIN dim_event e   ON e.event_id = b.event_id
    JOIN dim_student s ON s.student_id = b.student_id
""")

con.sql("SELECT * FROM dim_event").show()
con.sql("SELECT COUNT(*) AS students FROM dim_student").show()
con.sql("SELECT * FROM fact_ticket_sales LIMIT 5").show()

┌───────────┬──────────┬────────────────────────────┬────────────┬────────────┬──────────────────┐
│ event_key │ event_id │        event_title         │ event_club │ event_date │   event_venue    │
│   int64   │ varchar  │          varchar           │  varchar   │    date    │     varchar      │
├───────────┼──────────┼────────────────────────────┼────────────┼────────────┼──────────────────┤
│         1 │ EV-01    │ Robotics Demo Day          │ Robotics   │ 2026-10-15 │ Hall A           │
│         2 │ EV-02    │ Inter-faculty Debate Final │ Debate     │ 2026-10-22 │ Auditorium       │
│         3 │ EV-03    │ Autumn Concert             │ Music      │ 2026-10-29 │ Open-air theatre │
│         4 │ EV-04    │ Five-a-side Cup            │ Football   │ 2026-11-05 │ Sports field     │
│         5 │ EV-05    │ Blitz Chess Open           │ Chess      │ 2026-11-07 │ C7.201           │
│         6 │ EV-06    │ Line-follower Challenge    │ Robotics   │ 2026-11-12 │ Hall A           │
│         

In [28]:
# sanity check: a student id should map to one name/faculty, otherwise dim_student has duplicates
n_students = con.execute("SELECT COUNT(DISTINCT student_id) FROM bookings").fetchall()[0][0]
n_dim = con.execute("SELECT COUNT(*) FROM dim_student").fetchall()[0][0]
print("distinct student ids:", n_students, "| rows in dim_student:", n_dim)

distinct student ids: 103 | rows in dim_student: 103


In [29]:
fact_rows = con.execute("SELECT COUNT(*) FROM fact_ticket_sales").fetchall()[0][0]
assert fact_rows == len(booking_tickets)

fact_sum = con.execute("SELECT SUM(amount) FROM fact_ticket_sales").fetchall()[0][0]
assert abs(fact_sum - bookings["total"].sum()) < 0.001

print("fact rows:", fact_rows, "| sum amount:", fact_sum, "| sum total:", bookings["total"].sum())

fact rows: 314 | sum amount: 31285.0 | sum total: 31285.0


### Confirmed bookings: tickets and revenue per club and ticket type

In [30]:
con.sql("""
    SELECT
        e.event_club   AS club,
        f.ticket_type,
        SUM(f.qty)     AS tickets_sold,
        SUM(f.amount)  AS revenue
    FROM fact_ticket_sales f
    JOIN dim_event e ON e.event_key = f.event_key
    WHERE f.status = 'confirmed'
    GROUP BY club, f.ticket_type
    ORDER BY club, f.ticket_type
""").df()

,club,ticket_type,tickets_sold,revenue
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


### Save as Parquet

In [31]:
import os, shutil

if os.path.exists("star"):
    shutil.rmtree("star")
os.makedirs("star")

con.execute("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT PARQUET)")
con.execute("COPY dim_student TO 'star/dim_student.parquet' (FORMAT PARQUET)")
con.execute("""
    COPY (
        SELECT *, substr(CAST(date_key AS VARCHAR), 1, 4) || '-' || substr(CAST(date_key AS VARCHAR), 5, 2) AS booked_month
        FROM fact_ticket_sales
    )
    TO 'star/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))
""")

for root, dirs, files in sorted(os.walk("star")):
    for f in files:
        print(os.path.join(root, f))

star\dim_event.parquet
star\dim_student.parquet
star\fact_ticket_sales\booked_month=2026-09\data_0.parquet
star\fact_ticket_sales\booked_month=2026-10\data_0.parquet
star\fact_ticket_sales\booked_month=2026-11\data_0.parquet


In [32]:
# read it back to make sure nothing got lost
back = con.execute("SELECT COUNT(*), SUM(amount) FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning=true)").fetchall()[0]
print(back)

(314, 31285.0)
